In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)



from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

from analysis_village.cc1pi.var_configs import *
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_cathode_fix.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)

mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_evt_df = mc_bnb_evt_df[build_event_cumulative_masks(mc_bnb_evt_df, sideband = "")["energy"]]
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
cols_to_keep = [
    ('nu_categ', '', '', ''),
    ('genie_categ', '', '', ''),
    ('nu_categ_proton_reduced', '', '', '')
]
mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight.df", keys2load, 100)
#data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)

data_evt_df = data_df['cc1pi']
data_evt_df = data_evt_df[build_event_cumulative_masks(data_evt_df, sideband = "")["energy"]]
data_hdr_df = data_df['hdr']

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

In [ ]:
mc_evt_df = mc_bnb_evt_df
if "ar23p" in bnb_path:
    mc_evt_df = perform_truth_matching_low_memmory(mc_evt_df, mc_bnb_nu_df)
else:
    mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)   
print("Finished loading")    

n_mc = get_n_evt(mc_bnb_evt_df, True)
n_data = get_n_evt(data_evt_df, False)

print(f"{n_mc:<12} | {n_data:<12}")

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

# Register tqdm with pandas
tqdm.pandas(desc="Calculating Muon/Pion Vars")

def add_mu_pi_vars_column(pandora_df):
    # Fixed the list syntax and typos (removed ':' and fixed trk_end columns)
    EXPECTED_COLS = [
        'TL_mu','TL_pi',
        'dir_mu_x', 'dir_mu_y', 'dir_mu_z',
        'dir_pi_x', 'dir_pi_y', 'dir_pi_z',
        'trk_end_mu_x', 'trk_end_mu_y', 'trk_end_mu_z',
        'trk_end_pi_x', 'trk_end_pi_y', 'trk_end_pi_z','cos_theta_mu'
    ]
    
    group_levels = ['__ntuple', 'entry', 'rec.slc..index']
    
    # 1. Group and Apply
    muon_pion_vars = (
        pandora_df.groupby(level=group_levels, group_keys=False)
            .progress_apply(get_mu_pi_vars)
    )
    
    # Get all unique slice indices from the original DF
    all_slices_idx = pandora_df.index.droplevel('rec.slc.reco.pfp..index').unique()
    
    if muon_pion_vars.empty:
        slcdf = pd.DataFrame(index=all_slices_idx, columns=EXPECTED_COLS)
    else:
        # 2. Fix the duplicate index issue
        # This ensures we have one set of variables per slice
        if muon_pion_vars.index.duplicated().any():
            muon_pion_vars = muon_pion_vars[~muon_pion_vars.index.duplicated(keep='first')]

        if isinstance(muon_pion_vars, pd.Series):
            if not isinstance(muon_pion_vars.index, pd.MultiIndex):
                 muon_pion_vars = muon_pion_vars.to_frame().T
            else:
                 muon_pion_vars = muon_pion_vars.unstack()
                
        # Safe reindex for columns and then index
        slcdf = muon_pion_vars.reindex(columns=EXPECTED_COLS)
        slcdf = slcdf.reindex(all_slices_idx)
    
    # 3. Enforce schema
    for col in EXPECTED_COLS:
        slcdf[col] = pd.to_numeric(slcdf[col], errors='coerce').astype('float32')
            
    # 4. Apply MultiIndex columns
    slcdf.columns = pd.MultiIndex.from_tuples(
        [('slc', 'measure_var', col, '', '', '') for col in slcdf.columns]
    )
    
    # 5. Join back 
    # Use 'left' join on the index levels
    pandora_df = pandora_df.join(slcdf, on=group_levels)
    
    return pandora_df

def get_mu_pi_vars(group):
    # Safety check for empty or tiny groups
    if len(group) < 2:
        return pd.Series({col: -999.0 for col in [
            'dir_mu_x', 'dir_mu_y', 'dir_mu_z', 'dir_pi_x', 'dir_pi_y', 'dir_pi_z',
            'trk_end_mu_x', 'trk_end_mu_y', 'trk_end_mu_z', 'trk_end_pi_x', 'trk_end_pi_y', 'trk_end_pi_z', 
        'TL_mu', 'TL_pi',"cos_theta_mu"
        ]})

    # --- MUON SELECTION ---
    # Using try-except or check to handle potential CutMasks errors
    try:
        exiting_mask = CutMasks.exiting_pfp_mask(group)
    except:
        exiting_mask = np.zeros(len(group), dtype=bool)
    
    if exiting_mask.sum() >= 1:
        muon_row = group.loc[exiting_mask].iloc[[0]]
    else:
        # Sort by BDT score
        group_sorted = group.sort_values(('pfp','trk','bdt_muon_pion_score','','',''))
        muon_row = group_sorted.iloc[[-1]]
      
    # --- PION SELECTION ---
    remaining_pfps = group.drop(muon_row.index)
    pion_row = remaining_pfps.sort_values(('pfp','trk','len','','','')).iloc[[-1]]
    
    # Extract values
    return pd.Series({
        'TL_mu': muon_row.pfp.trk.len.iloc[0],
        'TL_pi': pion_row.pfp.trk.len.iloc[0],
        
        'dir_mu_x': muon_row.pfp.trk.dir.x.iloc[0],
        
        'dir_mu_x': muon_row.pfp.trk.dir.x.iloc[0],
        'dir_mu_y': muon_row.pfp.trk.dir.y.iloc[0],
        'dir_mu_z': muon_row.pfp.trk.dir.z.iloc[0],
        'dir_pi_x': pion_row.pfp.trk.dir.x.iloc[0],
        'dir_pi_y': pion_row.pfp.trk.dir.y.iloc[0],
        'dir_pi_z': pion_row.pfp.trk.dir.z.iloc[0],
        
        'trk_end_mu_x': muon_row.pfp.trk.end.x.iloc[0],
        'trk_end_mu_y': muon_row.pfp.trk.end.y.iloc[0],
        'trk_end_mu_z': muon_row.pfp.trk.end.z.iloc[0],
        
        'trk_end_pi_x': pion_row.pfp.trk.end.x.iloc[0],
        'trk_end_pi_y': pion_row.pfp.trk.end.y.iloc[0],
        'trk_end_pi_z': pion_row.pfp.trk.end.z.iloc[0],
        "cos_theta_mu":  muon_row.pfp.trk.dir.z.iloc[0],
    })


In [ ]:
# 1. Define the boolean mask for candidates
# We use data_evt_df instead of pandora_df here
candidate_mask = (
    data_evt_df.slc.cut.inside_FV & 
    data_evt_df.slc.cut.t0 & 
    data_evt_df.slc.cut.track & 
    CutMasks.is_MIP_candidate_mask(data_evt_df) & 
    data_evt_df.slc.cut.containment
)
# 3. Pass the candidate dataframe into your variable-adding function
# Note: Since the function returns the modified DF, we assign it back
data_evt_df = add_mu_pi_vars_column(data_evt_df[candidate_mask])

In [ ]:
candidate_mask = (
    mc_evt_df.slc.cut.inside_FV & 
    mc_evt_df.slc.cut.t0 & 
    mc_evt_df.slc.cut.track & 
    CutMasks.is_MIP_candidate_mask(mc_evt_df) & 
    mc_evt_df.slc.cut.containment
)
# 3. Pass the candidate dataframe into your variable-adding function
# Note: Since the function returns the modified DF, we assign it back
mc_evt_df = add_mu_pi_vars_column(mc_evt_df[candidate_mask])


In [ ]:
mc_evt_df.columns

In [ ]:
mask = mc_evt_df.slc.measure_var.reco_cos_theta_mu == mc_evt_df.slc.measure_var.cos_theta_mu

cols_to_show = [
    ('slc', 'measure_var', 'reco_cos_theta_mu', '', '', ''), 
    ('slc', 'measure_var', 'cos_theta_mu', '', '', '')
]

print(mc_evt_df.loc[mask, cols_to_show])

In [ ]:
'''
x_col = ('slc', 'measure_var', 'dir_mu_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_mu_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_mu', '', '', '')

# 2. Calculate using arctan2 (y, x)
# Note: typically phi is arctan2(y, x)
mc_evt_df[phi_col] = np.arctan(
    mc_evt_df[y_col]/
    mc_evt_df[x_col]
)

data_evt_df[phi_col] = np.arctan(
    data_evt_df[y_col]/
    data_evt_df[x_col]
)
'''
import numpy as np

# 1. Define the column keys
x_col = ('slc', 'measure_var', 'dir_mu_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_mu_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_mu', '', '', '')

# Convert MC to degrees
mc_evt_df[phi_col] = np.degrees(np.arctan2(mc_evt_df[y_col], mc_evt_df[x_col]))

# Convert Data to degrees
data_evt_df[phi_col] = np.degrees(np.arctan2(data_evt_df[y_col], data_evt_df[x_col]))



# 1. Define the column keys
x_col = ('slc', 'measure_var', 'dir_pi_x', '', '', '')
y_col = ('slc', 'measure_var', 'dir_pi_y', '', '', '')
phi_col = ('slc', 'measure_var', 'phi_pi', '', '', '')

# Convert MC to degrees
mc_evt_df[phi_col] = np.degrees(np.arctan2(mc_evt_df[y_col], mc_evt_df[x_col]))

# Convert Data to degrees
data_evt_df[phi_col] = np.degrees(np.arctan2(data_evt_df[y_col], data_evt_df[x_col]))


In [ ]:

R_mc = np.sqrt(mc_evt_df[x_col]**2 + mc_evt_df[y_col]**2)
print(f"Average Transverse Magnitude: {R_mc.mean()}")

In [ ]:
group_levels =  ['__ntuple','entry', 'rec.slc..index']

In [ ]:
print(mc_evt_df.slc.cut_var.n_MIP_candidates)
print(mc_evt_df.pfp.trk.len)
print(mc_evt_df[mc_evt_df.pfp.trk.len > 3].pfp.trk.len)

In [ ]:
n_mc = get_n_evt(mc_evt_df, True)
n_data = get_n_evt(data_evt_df, False)

print(f"{n_mc:<12} | {n_data:<12}")

n_mc_2 = get_n_evt(mc_evt_df[TPC_containment_mask(mc_evt_df,group_levels)], True)
n_data_2 = get_n_evt(data_evt_df[TPC_containment_mask(data_evt_df,group_levels)], False)

print(f"{n_mc_2:<12} | {n_data_2:<12}")
print(n_mc_2/n_mc)


HelperFunctions.print_purity(mc_evt_df, ('truth','nu_categ','','','',''))
HelperFunctions.print_purity(mc_evt_df[TPC_containment_mask(mc_evt_df,group_levels)], ('truth','nu_categ','','','',''))


In [ ]:
config_dir_mu_x = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','dir_mu_x','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-1, 1, 41),
    xlabel=r'dir_mu_x',
    ylabel=slices_y_label
)


config_phi_mu = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','phi_mu','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-180, 180, 41),
    xlabel=r'phi_mu',
    ylabel=slices_y_label
)

config_trk_end_x = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','trk_end_mu_x','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-200, 200, 41),
    xlabel=r'trk end x [cm]',
    ylabel=slices_y_label
)

config_trk_end_y = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','trk_end_mu_y','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-200, 200, 41),
    xlabel=r'trk end y [cm]',
    ylabel=slices_y_label
)

config_trk_end_z = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','trk_end_mu_z','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(0, 500, 21),
    xlabel=r'trk end z [cm]',
    ylabel=slices_y_label
)




mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0) & (df.slc.cut_var.n_exiting_pfps == 0),
    "TPC1": lambda df: (df.slc.vertex.x > 0) & (df.slc.cut_var.n_exiting_pfps == 0),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps == 0),
}

var_configs = [config_phi_mu, config_trk_end_x]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)



In [ ]:
def InFV(df):
    xmax = 190.
    zmin = 10.
    zmax = 450.
    ymax_highz = 100.
    pass_xz = (np.abs(df.slc.vertex.x) < xmax) & (df.slc.vertex.z > zmin) & (df.slc.vertex.z < zmax)
    pass_y = ((df.slc.vertex.z < 250) & (np.abs(df.slc.vertex.y) < 190.)) | ((df.slc.vertex.z > 250) & (df.slc.vertex.y > -190.) & (df.slc.vertex.y < ymax_highz))
    return pass_xz & pass_y


# General including all

In [ ]:

mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0),
    "TPC1": lambda df: (df.slc.vertex.x > 0),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
    "All": lambda df: df.slc.vertex.x > -99999,
    "New FV": lambda df: InFV(df),
    "New FV TPC contained": lambda df: InFV(df) & TPC_containment_mask(df,group_levels),
}

var_configs = [config_n_exiting_pfps, config_dir_mu_x, config_phi_mu, config_trk_end_x, config_trk_end_y, config_trk_end_z, config_vtx_x,config_vtx_y,config_vtx_z]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)

# Exiting only

In [ ]:
import numpy as np

# 1. Define the column keys
mu_z_col = ('slc', 'measure_var', 'dir_mu_z', '', '', '')
pi_z_col = ('slc', 'measure_var', 'dir_pi_z', '', '', '')


# 4. Add it back to the dataframe if you want to keep it
# I recommend using a simpler name for the new column
data_evt_df[('slc', 'measure_var', 'sum_sign_z', '', '', '')] = np.sign(data_evt_df[mu_z_col]) + np.sign(data_evt_df[pi_z_col])
mc_evt_df[('slc', 'measure_var', 'sum_sign_z', '', '', '')] = np.sign(mc_evt_df[mu_z_col]) + np.sign(mc_evt_df[pi_z_col])


In [ ]:
def exiting_pfp_mask(df):
    xmin = -200 + 10
    xmax = 200 - 10
    ymin = -200 + 10
    ymax = 200 - 10
    zmin = 10
    zmax = 500 - 10
    
    not_in_fv_start = (df.pfp.trk.start.x < xmin) | (df.pfp.trk.start.x > xmax) | (df.pfp.trk.start.y < ymin) | (df.pfp.trk.start.y > ymax) | (df.pfp.trk.start.z < zmin) | (df.pfp.trk.start.z  > zmax)
    not_in_fv_end = (df.pfp.trk.end.x < xmin) | (df.pfp.trk.end.x > xmax) | (df.pfp.trk.end.y < ymin) | (df.pfp.trk.end.y > ymax) | (df.pfp.trk.end.z < zmin) | (df.pfp.trk.end.z  > zmax)
    return not_in_fv_start | not_in_fv_end
    
def add_n_exiting_pfps_column(df):
    exiting_pfp_df = df[(exiting_pfp_mask(df))]
    exiting_pfp_counts = exiting_pfp_df.groupby(level=group_levels).size()

    target_key = ('slc','cut_var','n_exiting_pfps_10cm','','','')
    
    # Expand counts to full MultiIndex (broadcast to pfp level)
    df.loc[:,target_key] = df.index.droplevel('rec.slc.reco.pfp..index').map(exiting_pfp_counts)
    
    # Replace NaN (slices with 0 tracks) with 0
    df.loc[:,target_key] = df[target_key].fillna(0)
    
    return df


In [ ]:

mc_evt_df = add_n_exiting_pfps_column(mc_evt_df)
data_evt_df = add_n_exiting_pfps_column(data_evt_df)

In [ ]:


'''
mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0)& (df.slc.cut_var.n_exiting_pfps == 1),
    "TPC1": lambda df: (df.slc.vertex.x > 0)& (df.slc.cut_var.n_exiting_pfps == 1),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels)& (df.slc.cut_var.n_exiting_pfps == 1),
    "All": lambda df: (df.slc.vertex.x > -99999) & (df.slc.cut_var.n_exiting_pfps == 1),
    "New FV": lambda df: InFV(df)& (df.slc.cut_var.n_exiting_pfps == 1),
    "New FV TPC contained": lambda df: InFV(df)& (df.slc.cut_var.n_exiting_pfps == 1) & TPC_containment_mask(df,group_levels),
}
'''

config_n_exiting_pfps_10cm = FullHistogramConfig(
    file_name = "n_exiting_pfps_10cm", 
    var_evt_reco_col=('slc','cut_var','n_exiting_pfps_10cm','','',''),
    truth_column=nu_categ_column,
    first_per_slice = True,
    start_cut = "proton_BDT",
    end_cut = "michel",
    bins=np.linspace(0, 5, 6),
    xlabel=r'Number of exiting pfps 10 cm',
    ylabel= slices_y_label,
    cut_value = [2],
    clip = False
)


mask_dict = {
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
    "All": lambda df: (df.slc.vertex.x > -99999),
    "New FV": lambda df: InFV(df),
}

config_sumofsings = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','sum_sign_z','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(-2, 2, 4),
    xlabel=r'trk end y [cm]',
    ylabel=slices_y_label
)

var_configs = [config_n_exiting_pfps, config_n_exiting_pfps_10cm]

#config_sumofsings, config_dir_mu_x, config_phi_mu, config_trk_end_x, config_trk_end_y, config_trk_end_z, config_angle_between_candidates, 
 #                  config_vtx_x,config_vtx_y,config_vtx_z, ]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)

# Remove high/low y

In [ ]:

mask_dict = {
    "TPC0": lambda df: (df.slc.vertex.x < 0)& (df.slc.cut_var.n_exiting_pfps == 1) & (200 - abs(df.slc.measure_var.trk_end_mu_y) > 5),
    "TPC1": lambda df: (df.slc.vertex.x > 0)& (df.slc.cut_var.n_exiting_pfps == 1) & (200 - abs(df.slc.measure_var.trk_end_mu_y) > 5),
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels)& (df.slc.cut_var.n_exiting_pfps == 1)  & (200 - abs(df.slc.measure_var.trk_end_mu_y) > 5),
    "All": lambda df: (df.slc.vertex.x > -99999) & (df.slc.cut_var.n_exiting_pfps == 1)  & (200 - abs(df.slc.measure_var.trk_end_mu_y) > 5),
    "New FV": lambda df: InFV(df)& (df.slc.cut_var.n_exiting_pfps == 1)  & (200 - abs(df.slc.measure_var.trk_end_mu_y) > 5),
    "New FV TPC contained": lambda df: InFV(df)& (df.slc.cut_var.n_exiting_pfps == 1) & TPC_containment_mask(df,group_levels),
}

var_configs = [config_n_exiting_pfps, config_dir_mu_x, config_phi_mu, config_trk_end_x, config_trk_end_y, config_trk_end_z, config_vtx_x,config_vtx_y,config_vtx_z]
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)

In [ ]:

mask_dict = {
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
    "TPC_contained_only_contained": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps == 0),
    "All": lambda df: df.slc.vertex.x > -99999,
    "All_contained": lambda df: df.slc.cut_var.n_exiting_pfps == 0,
    "New FV": lambda df: InFV(df),
    "New FV TPC Contained": lambda df: InFV(df) & TPC_containment_mask(df,group_levels),
    "New FV ALL Contained": lambda df: InFV(df) & df.slc.cut_var.n_exiting_pfps == 0,
}


config_TL_mu = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','TL_mu','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(0, 100, 11),
    xlabel=r'TL mu [cm]',
    ylabel=slices_y_label
)



config_TL_pi = FullHistogramConfig(
    file_name = "",      
    var_evt_reco_col=('slc','measure_var','TL_pi','','',''),
    truth_column = nu_categ_column,
    first_per_slice = True,
    start_cut = "energy",
    end_cut = "energy",
    bins=np.linspace(0, 100, 11),
    xlabel=r'TL pi [cm]',
    ylabel=slices_y_label
)



var_configs = final_var_configs #[config_angle_between_candidates, config_nu_score, config_bc_flash_score]#
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from IPython.display import display
import numpy as np

for var_config in var_configs:
    n_masks = len(mask_dict)

    original_show = plt.show
    plt.show = lambda: None

    figs = []
    for name, mask_func in mask_dict.items():
        fig, red_chi2 = plot_stacked_histogram_with_ratio(
            mc_df=mc_evt_df[mask_func(mc_evt_df)],
            data_df=data_evt_df[mask_func(data_evt_df)],
            config=var_config,
            cov_frac_matrix=None, cov_matrix=None,
            title="", weight_column=pot_weight_col,
            data_pot=data_tot_pot, normalize=False,
            show_stats=False, symmetric_ratio=True, divide_by_bin_width=False
        )
        figs.append((fig, name))

    plt.show = original_show

    n_cols = min(3, n_masks)
    n_rows = int(np.ceil(n_masks / n_cols))

    grid_fig = plt.figure(figsize=(10 * n_cols, 8 * n_rows))

    for idx, (sub_fig, name) in enumerate(figs):
        canvas = FigureCanvasAgg(sub_fig)
        canvas.draw()
        img = np.asarray(canvas.buffer_rgba())

        ax = grid_fig.add_subplot(n_rows, n_cols, idx + 1)
        ax.imshow(img)
        ax.set_title(name, fontsize=14, fontweight='bold', pad=8)
        ax.axis('off')
        plt.close(sub_fig)

    grid_fig.suptitle(var_config.xlabel, fontsize=16, fontweight='bold', y=1.01)
    grid_fig.tight_layout()
    display(grid_fig)       # <-- replaces plt.show()
    plt.close(grid_fig)



In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import matplotlib.colors as mcolors

def plot_2d_hist_data_mc(
    mc_df, data_df,
    var_x, var_y,
    bins_x, bins_y,
    xlabel, ylabel,
    mask_name="",
    weight_column=None,
    data_pot=None,
):
    wgt_col = ('slc', 'wgt', '', '', '', '')
    fig, (ax_mc, ax_data) = plt.subplots(1, 2, figsize=(14, 5))

    # --- MC ---
    mc_x = mc_df[var_x].dropna()
    mc_y = mc_df[var_y].dropna()
    common_idx = mc_x.index.intersection(mc_y.index)
    mc_x = mc_x.loc[common_idx]
    mc_y = mc_y.loc[common_idx]
    mc_w = mc_df[weight_column].loc[common_idx] if weight_column in mc_df.columns else np.ones(len(mc_x))
    mc_wgt = mc_df[wgt_col].loc[common_idx] if wgt_col in mc_df.columns else np.ones(len(mc_x))
    mc_w = mc_w * mc_wgt

    # --- Data ---
    data_x = data_df[var_x].dropna()
    data_y = data_df[var_y].dropna()
    common_idx_d = data_x.index.intersection(data_y.index)
    data_x = data_x.loc[common_idx_d]
    data_y = data_y.loc[common_idx_d]
    data_w = data_df[weight_column].loc[common_idx_d] if weight_column in data_df.columns else np.ones(len(data_x))
    data_wgt = data_df[wgt_col].loc[common_idx_d] if wgt_col in data_df.columns else np.ones(len(data_x))
    data_w = data_w * data_wgt

    h_mc, _, _   = np.histogram2d(mc_x,   mc_y,   bins=[bins_x, bins_y], weights=mc_w)
    h_data, _, _ = np.histogram2d(data_x, data_y, bins=[bins_x, bins_y], weights=data_w)

    # --- Area normalize ---
    mc_sum   = h_mc.sum()
    data_sum = h_data.sum()


    h_mc   = h_mc   / mc_sum   if mc_sum   > 0 else h_mc
    h_data = h_data / data_sum if data_sum > 0 else h_data

    
    h_mc   = np.ma.masked_where(h_mc == 0, h_mc)
    h_data = np.ma.masked_where(h_data == 0, h_data)
    
    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color='white')

    vmax = max(h_mc.max(), h_data.max()) if (h_mc.count() > 0 or h_data.count() > 0) else 1

    def plot_h2(ax, h, title):
        im = ax.pcolormesh(bins_x, bins_y, h.T, cmap=cmap, vmin=0, vmax=vmax)
        ax.set_xlabel(xlabel, fontsize=13)
        ax.set_ylabel(ylabel, fontsize=13)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.set_facecolor('white')
        plt.colorbar(im, ax=ax)

    pot_str = f" (POT={data_pot:.2e})" if data_pot else ""
    plot_h2(ax_mc,   h_mc,   f"MC — {mask_name}{pot_str}")
    plot_h2(ax_data, h_data, f"Data — {mask_name}{pot_str}")

    fig.tight_layout()
    return fig
    

# --- Loop over masks ---

var_x     = ('slc','measure_var','trk_end_mu_x','','','')
var_y     = ('slc','measure_var','trk_end_mu_y','','','')
'''
var_x     = ('pfp','trk','end','x','','')
var_y     = ('pfp','trk','end','y','','')
'''
bins_x    = np.linspace(-200, 200, 41)
bins_y    = np.linspace(-200, 200, 41)


slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
mask_dict = {
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
    "TPC_contained_exiting": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps == 1),
    "All": lambda df: df.slc.vertex.x > -99999,
    "All_exiting": lambda df: df.slc.cut_var.n_exiting_pfps == 1,
    "New FV": lambda df: InFV(df),
    "New FV Exiting": lambda df: InFV(df) & (df.slc.cut_var.n_exiting_pfps == 1),
}

for name, mask_func in mask_dict.items():
    fig = plot_2d_hist_data_mc(
        mc_df=mc_evt_df[mask_func(mc_evt_df)].groupby(level=slice_levels, sort=False).first(),
        data_df=data_evt_df[mask_func(data_evt_df)].groupby(level=slice_levels, sort=False).first(),
        var_x=var_x, var_y=var_y,
        bins_x=bins_x, bins_y=bins_y,
        xlabel='trk end $\\mu$ x [cm]',
        ylabel='trk end $\\mu$ y [cm]',
        mask_name=name,
        weight_column=pot_weight_col,
        data_pot=data_tot_pot,
    )
    display(fig)
    plt.close(fig)

In [ ]:
#mask = TPC_containment_mask(data_evt_df,group_levels) & (data_evt_df.slc.cut_var.n_exiting_pfps == 1) & ((200 - abs(data_evt_df.slc.measure_var.trk_end_mu_y) < 5) | (200 - abs(data_evt_df.slc.measure_var.trk_end_mu_x) < 5))
mask = TPC_containment_mask(data_evt_df,group_levels) & (data_evt_df.slc.cut_var.n_exiting_pfps == 1) & ((200 - abs(data_evt_df.slc.measure_var.trk_end_mu_x) < 5)) & ( data_evt_df.slc.measure_var.trk_end_mu_y < 0)

# 1. Get the unique indices from your filtered and grouped event dataframe
# We use .index.unique() to get the specific (__ntuple, __entry) pairs
# 1. Filter the dataframe first
# 1. Filter the dataframe
filtered_df = data_evt_df[mask]

# 2. Drop the sub-levels (slc and pfp) so only __ntuple and entry remain
# Then get the unique pairs
filtered_indices = filtered_df.index.droplevel(['rec.slc..index', 'rec.slc.reco.pfp..index']).unique()

# 3. Filter the header dataframe
data_hdr_df_filtered = data_hdr_df[data_hdr_df.index.isin(filtered_indices)]

# 4. Show the result
print(f"Unique events found: {len(filtered_indices)}")
print(data_hdr_df_filtered[['run', 'subrun','evt']]) # Assuming these cols exist in hdr



# Define the output path
output_dir = "/exp/sbnd/data/users/lpelegri/DisplaysTitus"
output_file = os.path.join(output_dir, "event_list_for_samweb.txt")

# Ensure directory exists
os.makedirs(output_dir, exist_ok=True)
pd.set_option('display.max_rows', 500)
# 1. Get unique Run, Subrun, Event
# We drop duplicates to ensure we don't scan the same file twice
event_list = data_hdr_df_filtered[['run', 'subrun', 'evt']].drop_duplicates()

# 2. Save to a space-separated text file (easy for bash to read)
# We use header=False so the bash script doesn't try to process the column names
#event_list.to_csv(output_file, sep=' ', index=False, header=False)

print(f"Successfully saved {len(event_list)} events to: {output_file}")

'''
13       33     18255       1  221871
57       37     18255       1  499091
125      2      18255       1  272719
368      16     18255       1  323536
384      11     18255       1  271770
'''

In [ ]:
import numpy as np

# 1. Get your list of unique events to loop through
event_list = data_hdr_df_filtered[['run', 'subrun', 'evt']].drop_duplicates()

# 2. Define the columns you want to see
cols_to_print = [
    ('slc', 'vertex', 'x', '', '', ''),
    ('slc', 'vertex', 'y', '', '', ''),
    ('slc', 'vertex', 'z', '', '', ''),
    ('slc', 'measure_var', 'trk_end_mu_x', '', '', ''),
    ('slc', 'measure_var', 'trk_end_mu_y', '', '', ''),
    ('slc', 'measure_var', 'trk_end_mu_z', '', '', ''),
    ('slc', 'measure_var', 'trk_end_pi_x', '', '', ''),
    ('slc', 'measure_var', 'trk_end_pi_z', '', '', '')
]

print(f"Printing details for {len(event_list)} events...")
print("="*80)

# 3. Start the loop
for index, row in event_list.iterrows():
    t_run, t_subrun, t_evt = row['run'], row['subrun'], row['evt']
    
    # Get the internal index from the header
    hdr_match = data_hdr_df[
        (data_hdr_df['run'] == t_run) & 
        (data_hdr_df['subrun'] == t_subrun) & 
        (data_hdr_df['evt'] == t_evt)
    ]
    
    if hdr_match.empty:
        continue
        
    matched_indices = hdr_match.index
    
    # 4. Filter data_evt_df
    # We use the droplevel approach to match the 2-level header index
    mask = data_evt_df.index.droplevel(['rec.slc..index', 'rec.slc.reco.pfp..index']).isin(matched_indices)
    event_details = data_evt_df.loc[mask, cols_to_print].head(1)
    
    if not event_details.empty:
        # Calculate the sum_sign_z on the fly for this specific event
         # Print Header for the event
        print(f"RUN: {t_run} | SUBRUN: {t_subrun} | EVENT: {t_evt}")
        
        # Print the physics variables
        print(event_details.to_string())
        print("-" * 80)

print("Loop Complete.")

In [ ]:
#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_evt_all_df = mc_bnb_df['cc1pi']
mc_bnb_evt_all_df = mc_bnb_evt_all_df[build_event_cumulative_masks(mc_bnb_evt_all_df, sideband = "")["nu_score"]]

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_evt_all_df = data_df['cc1pi']
data_evt_all_df = data_evt_all_df[build_event_cumulative_masks(data_evt_all_df, sideband = "")["nu_score"]]

In [ ]:
mask_dict = {
    "TPC_contained": lambda df: TPC_containment_mask(df,group_levels),
    "TPC_contained_exiting": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps >= 1),
    "TPC_contained_contained": lambda df: TPC_containment_mask(df,group_levels) & (df.slc.cut_var.n_exiting_pfps == 0),
    "All": lambda df: df.slc.vertex.x > -99999,
    "All_exiting": lambda df: df.slc.cut_var.n_exiting_pfps >= 1,
    "All_contained": lambda df: df.slc.cut_var.n_exiting_pfps == 0,
    "New FV": lambda df: InFV(df),
    "New FV Exiting": lambda df: InFV(df) & (df.slc.cut_var.n_exiting_pfps >= 1),
    "New FV Contained": lambda df: InFV(df) & (df.slc.cut_var.n_exiting_pfps == 0),
}

# --- Loop over masks ---
var_x     = ('pfp','trk','end','x','','')
var_y     = ('pfp','trk','end','y','','')
bins_x    = np.linspace(-200, 200, 41)
bins_y    = np.linspace(-200, 200, 41)


for name, mask_func in mask_dict.items():
    fig = plot_2d_hist_data_mc(
        mc_df=mc_bnb_evt_all_df[mask_func(mc_bnb_evt_all_df)],
        data_df=data_evt_all_df[mask_func(data_evt_all_df)],
        var_x=var_x, var_y=var_y,
        bins_x=bins_x, bins_y=bins_y,
        xlabel='trk end x [cm]',
        ylabel='trk end y [cm]',
        mask_name=name,
        weight_column=pot_weight_col,
        data_pot=data_tot_pot,
    )
    display(fig)
    plt.close(fig)